# Task 1 (Easy) – Data Cleaning & Preprocessing: Netflix Dataset

**Goal:** Prepare the raw Netflix titles dataset for analysis by cleaning, transforming and organizing it.


## Step 1 – Import the dataset

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = "Dataset.csv"
df_raw = pd.read_csv(r"D:\Ausify_internship_task\data\Dataset.csv")
df = df_raw.copy()          # work on a copy, keep the raw data untouched

print("Shape:", df.shape)
df.head()

Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


In [3]:
# Summary of the numeric column and the text columns
display(df.describe())
display(df.describe(include="all").T)

,release_year
count,8790.000000
mean,2014.183163
std,8.825466
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8790,8790,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8790,2,Movie,6126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8790,8787,9-Feb,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,8790,4528,Not Given,2588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,8790,86,United States,3240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8790,1713,1/1/2020,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8790.0,NaN,NaN,NaN,2014.183163,8.825466,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8790,14,TV-MA,3205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8790,220,1 Season,1791,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listed_in,8790,513,"Dramas, International Movies",362,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**First impressions**
- Every column is stored as text except `release_year`.
- `date_added` is a string (`m/d/yyyy`), not a real date.
- `duration` mixes two units (`90 min` for movies, `2 Seasons` for TV shows).
- `director`, `listed_in` hold several comma-separated values in one cell.

## Step 2 – Identify missing and duplicate records

### 2.1 Standard missing values (`NaN`)

In [4]:
missing = df.isnull().sum().to_frame("missing_count")
missing["missing_%"] = (missing["missing_count"] / len(df) * 100).round(2)
missing

,missing_count,missing_%
show_id,0,0.0
type,0,0.0
title,0,0.0
director,0,0.0
country,0,0.0
date_added,0,0.0
release_year,0,0.0
rating,0,0.0
duration,0,0.0
listed_in,0,0.0


Pandas reports **zero** nulls – but that can be misleading. Missing data is often hidden behind
placeholder text such as `"Not Given"`, `"Unknown"`, `"N/A"` or empty strings. Let's check for those.

In [5]:
PLACEHOLDERS = {"not given", "unknown", "n/a", "na", "none", "nan", "null", "-", ""}

hidden = {}
for col in df.select_dtypes(exclude="number").columns:
    mask = df[col].astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
    if mask.sum():
        hidden[col] = int(mask.sum())

hidden_df = pd.Series(hidden, name="hidden_missing_count").to_frame()
hidden_df["hidden_missing_%"] = (hidden_df["hidden_missing_count"] / len(df) * 100).round(2)
hidden_df

,hidden_missing_count,hidden_missing_%
title,1,0.01
director,2588,29.44
country,287,3.27


In [6]:
# Which placeholder values are they, exactly?
for col in ["director", "country"]:
    print(col, "->", df.loc[df[col].str.strip().str.lower().isin(PLACEHOLDERS), col].unique())

# 'Unknown' in title is a genuine movie title, not a missing value
print(df.loc[df["title"].str.lower() == "unknown", ["show_id", "title", "type", "release_year"]])

director -> <StringArray>
['Not Given']
Length: 1, dtype: str
country -> <StringArray>
['Not Given']
Length: 1, dtype: str
     show_id    title   type  release_year
1336   s1468  Unknown  Movie          2011


**Finding:** `director` (~29%) and `country` (~3%) contain the placeholder **"Not Given"** – these are real
missing values that need treating. The single `"Unknown"` in `title` is an actual movie title, so it is **kept**.

### 2.2 Duplicate records

In [7]:
print("Exact duplicate rows            :", df.duplicated().sum())
print("Duplicate show_id               :", df["show_id"].duplicated().sum())

# show_id is just a row key; the same title can be listed twice under different IDs
content_cols = [c for c in df.columns if c != "show_id"]
print("Duplicates ignoring show_id     :", df.duplicated(subset=content_cols).sum())

Exact duplicate rows            : 0
Duplicate show_id               : 0
Duplicates ignoring show_id     : 3


In [8]:
# Look at them
df[df.duplicated(subset=content_cols, keep=False)].sort_values("title")

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"


In [9]:
# Same title appearing more than once - are they duplicates or different shows?
same_title = df[df["title"].duplicated(keep=False)].sort_values("title")
same_title[["show_id", "type", "title", "country", "release_year", "duration"]]

,show_id,type,title,country,release_year,duration
2925,s3963,Movie,15-Aug,India,2019,124 min
4261,s5967,Movie,15-Aug,India,2019,124 min
3285,s4523,Movie,22-Jul,Norway,2018,144 min
4260,s5966,Movie,22-Jul,Norway,2018,144 min
393,s3997,TV Show,9-Feb,Pakistan,2018,1 Season
537,s5965,TV Show,9-Feb,Pakistan,2018,1 Season


**Finding:** No fully identical rows, but **3 records are duplicates** that differ only in `show_id`
(identical in every other column). Titles like `9-Feb`, `15-Aug` and `22-Jul` also repeat,
yet those are *different* titles (different type/country/year), so they are **not** duplicates.

### 2.3 Other quality checks

In [10]:
# Whitespace / formatting problems in text columns
text_cols = df.select_dtypes(exclude="number").columns
issues = {}
for c in text_cols:
    s = df[c]
    issues[c] = {
        "leading/trailing spaces": int((s != s.str.strip()).sum()),
        "non-breaking spaces": int(s.str.contains("\u00a0", regex=False).sum()),
        "double spaces": int(s.str.contains(r"\s{2,}", regex=True).sum()),
    }
pd.DataFrame(issues).T

,leading/trailing spaces,non-breaking spaces,double spaces
show_id,0,0,0
type,0,0,0
title,1,5,3
director,0,0,0
country,0,0,0
date_added,0,0,0
rating,0,0,0
duration,0,0,0
listed_in,0,0,0


In [11]:
print(df.loc[df["title"].str.contains("\u00a0|\\s{2,}", regex=True), "title"].tolist())

['A Second Chance:  Rivals!', 'Legend\xa0of\xa0Exorcism', 'Operation Varsity Blues:\xa0The\xa0College\xa0Admissions\xa0Scandal', 'Up Among  The Stars', 'Chaar\xa0Sahibzaade', 'Consequences\xa0', 'Tremors 5:  Bloodline', 'Akbar\xa0Birbal']


In [12]:
# Categorical value sets
print(df["type"].value_counts(), "\n")
print(df["rating"].value_counts())

type
Movie      6126
TV Show    2664
Name: count, dtype: int64 



rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [13]:
# Date format check: does every date_added parse with m/d/yyyy?
parsed = pd.to_datetime(df["date_added"], format="%m/%d/%Y", errors="coerce")
print("Unparseable dates:", parsed.isna().sum())
print("Date range:", parsed.min().date(), "->", parsed.max().date())

# Logical check: a title can't normally be added before it was released
print("Rows added to Netflix in a year BEFORE release_year:", (parsed.dt.year < df["release_year"]).sum())

Unparseable dates: 0
Date range: 2008-01-01 -> 2021-09-25
Rows added to Netflix in a year BEFORE release_year: 14


## Step 3 – Handle null values and inconsistent formats

### 3.1 Fix text formatting (whitespace, non-breaking spaces)

In [14]:
for c in df.select_dtypes(exclude="number").columns:
    df[c] = (
        df[c]
        .str.replace("\u00a0", " ", regex=False)   # non-breaking space -> normal space
        .str.replace(r"\s+", " ", regex=True)       # collapse repeated whitespace
        .str.strip()                               # trim ends
    )

print("Titles with odd whitespace left:", df["title"].str.contains("\u00a0|\\s{2,}|^\\s|\\s$", regex=True).sum())

Titles with odd whitespace left: 0


### 3.2 Remove duplicate records

In [15]:
before = len(df)
df = df.drop_duplicates(subset=content_cols, keep="first").reset_index(drop=True)
print(f"Rows before: {before:,} | after: {len(df):,} | removed: {before - len(df)}")

Rows before: 8,790 | after: 8,786 | removed: 4


> Whitespace was fixed *before* de-duplicating, so titles that differed only by a stray space (e.g. `"Consequences "`) were caught as well. That is why **4** rows were removed instead of the 3 found in Step 2.

In [16]:
# Re-check after the whitespace fix (it can reveal more duplicates)
print("Remaining duplicates:", df.duplicated(subset=content_cols).sum())

Remaining duplicates: 0


### 3.3 Treat missing values
**Strategy**

| Column | Issue | Treatment | Why |
|---|---|---|---|
| `director` | "Not Given" (~29%) | → `NaN` → fill with **"Unknown"** | Too many to drop; director can't be inferred |
| `country` | "Not Given" (~3%) | → `NaN` → fill with **"Unknown"** | Keeps the row for other analyses |

I also keep two boolean flags (`director_missing`, `country_missing`) so analysts can always tell imputed rows apart.

In [17]:
# Convert the placeholder to a real NaN first so we are explicit about what is missing
for col in ["director", "country"]:
    df[col] = df[col].replace({"Not Given": np.nan})

print(df[["director", "country"]].isnull().sum())

director    2587
country      287
dtype: int64


In [18]:
# Keep a record of which values were originally missing, then impute
df["director_missing"] = df["director"].isnull()
df["country_missing"]  = df["country"].isnull()

df["director"] = df["director"].fillna("Unknown")
df["country"]  = df["country"].fillna("Unknown")

print("Nulls left:", df.isnull().sum().sum())

Nulls left: 0


### 3.4 Standardize inconsistent categories (`rating`)

In [19]:
# Group the 14 rating codes into broader, easier-to-analyse audience categories
rating_map = {
    "TV-Y": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-Y7": "Older Kids", "TV-Y7-FV": "Older Kids", "TV-PG": "Older Kids", "PG": "Older Kids",
    "TV-14": "Teens", "PG-13": "Teens",
    "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults",
    "NR": "Not Rated", "UR": "Not Rated",
}
df["rating_category"] = df["rating"].map(rating_map)

# every rating must be mapped
assert df["rating_category"].notnull().all(), "Unmapped rating found!"
df["rating_category"].value_counts()

rating_category
Adults        4005
Teens         2645
Older Kids    1487
Kids           567
Not Rated       82
Name: count, dtype: int64

## Step 4 – Transform categorical and date-related columns

### 4.1 Date column → `datetime` and date features

In [20]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%m/%d/%Y")

df["year_added"]    = df["date_added"].dt.year
df["month_added"]   = df["date_added"].dt.month
df["month_name"]    = df["date_added"].dt.month_name()
df["day_of_week"]   = df["date_added"].dt.day_name()
df["quarter_added"] = df["date_added"].dt.quarter

df[["date_added", "year_added", "month_added", "month_name", "day_of_week", "quarter_added"]].head()

,date_added,year_added,month_added,month_name,day_of_week,quarter_added
0,2021-09-25,2021,9,September,Saturday,3
1,2021-09-24,2021,9,September,Friday,3
2,2021-09-24,2021,9,September,Friday,3
3,2021-09-22,2021,9,September,Wednesday,3
4,2021-09-24,2021,9,September,Friday,3


### 4.2 Split `duration` into numeric columns

In [21]:
# 'duration' mixes minutes (movies) and seasons (TV shows) -> split into two clean numeric columns
dur_num = df["duration"].str.extract(r"(\d+)")[0].astype("Int64")

df["duration_minutes"] = dur_num.where(df["type"] == "Movie")      # movies only
df["seasons"]          = dur_num.where(df["type"] == "TV Show")    # TV shows only

# validation: the unit text must agree with the type
assert df.loc[df["type"] == "Movie", "duration"].str.contains("min").all()
assert df.loc[df["type"] == "TV Show", "duration"].str.contains("Season").all()

df[["type", "duration", "duration_minutes", "seasons"]].sample(6, random_state=7)

,type,duration,duration_minutes,seasons
3543,Movie,101 min,101,<NA>
2658,Movie,106 min,106,<NA>
940,Movie,107 min,107,<NA>
5493,Movie,109 min,109,<NA>
5388,Movie,137 min,137,<NA>
4836,Movie,74 min,74,<NA>


### 4.3 Multi-valued columns (`listed_in`, `director`)

In [22]:
df = df.rename(columns={"listed_in": "genres"})

# genres: "Dramas, International Movies" -> count + primary genre
df["num_genres"]    = df["genres"].str.split(", ").str.len()
df["primary_genre"] = df["genres"].str.split(", ").str[0]

# directors: count (Unknown -> 0)
df["num_directors"] = np.where(
    df["director"] == "Unknown", 0, df["director"].str.split(", ").str.len()
)

df[["genres", "num_genres", "primary_genre", "director", "num_directors"]].head()

,genres,num_genres,primary_genre,director,num_directors
0,Documentaries,1,Documentaries,Kirsten Johnson,1
1,"Crime TV Shows, International TV Shows, TV Act...",3,Crime TV Shows,Julien Leclercq,1
2,"TV Dramas, TV Horror, TV Mysteries",3,TV Dramas,Mike Flanagan,1
3,"Children & Family Movies, Comedies",2,Children & Family Movies,Bruno Garotti,1
4,"Dramas, Independent Movies, International Movies",3,Dramas,Haile Gerima,1


### 4.4 Additional engineered features

In [23]:
# How many years passed between release and arrival on Netflix?
df["years_to_netflix"] = df["year_added"] - df["release_year"]

# Flag the logical anomaly found in Step 2.3 (added to Netflix before the release year).
# These are kept (they are plausible pre-release listings) but flagged for analysts.
df["added_before_release"] = df["years_to_netflix"] < 0

# Decade of release - handy for trend plots
df["release_decade"] = (df["release_year"] // 10 * 10).astype(str) + "s"

# Movie length bucket
df["movie_length_category"] = pd.cut(
    df["duration_minutes"],
    bins=[0, 60, 90, 120, np.inf],
    labels=["Short (<60 min)", "Medium (60-90 min)", "Long (90-120 min)", "Very long (>120 min)"],
)

print("Flagged as added before release:", int(df["added_before_release"].sum()))
df[["title", "release_year", "year_added", "years_to_netflix", "release_decade", "movie_length_category"]].head()

Flagged as added before release: 14


,title,release_year,year_added,years_to_netflix,release_decade,movie_length_category
0,Dick Johnson Is Dead,2020,2021,1,2020s,Medium (60-90 min)
1,Ganglands,2021,2021,0,2020s,NaN
2,Midnight Mass,2021,2021,0,2020s,NaN
3,Confessions of an Invisible Girl,2021,2021,0,2020s,Long (90-120 min)
4,Sankofa,1993,2021,28,1990s,Very long (>120 min)


### 4.5 Set memory-efficient categorical dtypes

In [24]:
cat_cols = ["type", "rating", "rating_category", "country", "primary_genre",
            "month_name", "day_of_week", "release_decade"]
for c in cat_cols:
    df[c] = df[c].astype("category")

# put month / weekday in natural order instead of alphabetical
months = ["January","February","March","April","May","June","July","August","September","October","November","December"]
days   = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df["month_name"]  = pd.Categorical(df["month_name"],  categories=months, ordered=True)
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=days,   ordered=True)

# tidy column order and drop the now-redundant original duration text
df = df.drop(columns=["duration"])
ordered_cols = [
    "show_id", "type", "title", "director", "num_directors", "director_missing",
    "country", "country_missing", "genres", "primary_genre", "num_genres",
    "rating", "rating_category", "release_year", "release_decade",
    "date_added", "year_added", "quarter_added", "month_added", "month_name", "day_of_week",
    "years_to_netflix", "added_before_release",
    "duration_minutes", "movie_length_category", "seasons",
]
df = df[ordered_cols]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8786 entries, 0 to 8785
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   show_id                8786 non-null   str           
 1   type                   8786 non-null   category      
 2   title                  8786 non-null   str           
 3   director               8786 non-null   str           
 4   num_directors          8786 non-null   int64         
 5   director_missing       8786 non-null   bool          
 6   country                8786 non-null   category      
 7   country_missing        8786 non-null   bool          
 8   genres                 8786 non-null   str           
 9   primary_genre          8786 non-null   category      
 10  num_genres             8786 non-null   int64         
 11  rating                 8786 non-null   category      
 12  rating_category        8786 non-null   category      
 13  release_year  

## Step 5 – Final validation and clean dataset

In [25]:
checks = {
    "No duplicate show_id":             not df["show_id"].duplicated().any(),
    "No duplicate content":             not df.duplicated(subset=["type","title","director","country","release_year","date_added"]).any(),
    "No null in core columns":          df[["show_id","type","title","director","country","date_added","release_year","rating"]].notnull().all().all(),
    "No 'Not Given' placeholders left": not (df[["director","country"]] == "Not Given").any().any(),
    "date_added is datetime":           pd.api.types.is_datetime64_any_dtype(df["date_added"]),
    "Movies have minutes, no seasons":  df.loc[df["type"]=="Movie","duration_minutes"].notnull().all() and df.loc[df["type"]=="Movie","seasons"].isnull().all(),
    "TV shows have seasons, no minutes":df.loc[df["type"]=="TV Show","seasons"].notnull().all() and df.loc[df["type"]=="TV Show","duration_minutes"].isnull().all(),
}
pd.Series(checks, name="passed").to_frame()

,passed
No duplicate show_id,True
No duplicate content,True
No null in core columns,True
No 'Not Given' placeholders left,True
date_added is datetime,True
"Movies have minutes, no seasons",True
"TV shows have seasons, no minutes",True


In [26]:
# Before vs After
summary = pd.DataFrame({
    "Raw":   [df_raw.shape[0], df_raw.shape[1], len(df_raw) - len(df),
              int((df_raw[["director","country"]] == "Not Given").sum().sum())],
    "Clean": [df.shape[0], df.shape[1], 0, 0],
}, index=["Rows", "Columns", "Duplicate records", "'Not Given' placeholders"])
summary

,Raw,Clean
Rows,8790,8786
Columns,10,26
Duplicate records,4,0
'Not Given' placeholders,2875,0


In [27]:
# Quick look at the finished data
df.head(10)

,show_id,type,title,director,num_directors,director_missing,country,country_missing,genres,primary_genre,num_genres,rating,rating_category,release_year,release_decade,date_added,year_added,quarter_added,month_added,month_name,day_of_week,years_to_netflix,added_before_release,duration_minutes,movie_length_category,seasons
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,1,False,United States,False,Documentaries,Documentaries,1,PG-13,Teens,2020,2020s,2021-09-25,2021,3,9,September,Saturday,1,False,90,Medium (60-90 min),<NA>
1,s3,TV Show,Ganglands,Julien Leclercq,1,False,France,False,"Crime TV Shows, International TV Shows, TV Act...",Crime TV Shows,3,TV-MA,Adults,2021,2020s,2021-09-24,2021,3,9,September,Friday,0,False,<NA>,NaN,1
2,s6,TV Show,Midnight Mass,Mike Flanagan,1,False,United States,False,"TV Dramas, TV Horror, TV Mysteries",TV Dramas,3,TV-MA,Adults,2021,2020s,2021-09-24,2021,3,9,September,Friday,0,False,<NA>,NaN,1
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,1,False,Brazil,False,"Children & Family Movies, Comedies",Children & Family Movies,2,TV-PG,Older Kids,2021,2020s,2021-09-22,2021,3,9,September,Wednesday,0,False,91,Long (90-120 min),<NA>
4,s8,Movie,Sankofa,Haile Gerima,1,False,United States,False,"Dramas, Independent Movies, International Movies",Dramas,3,TV-MA,Adults,1993,1990s,2021-09-24,2021,3,9,September,Friday,28,False,125,Very long (>120 min),<NA>
5,s9,TV Show,The Great British Baking Show,Andy Devonshire,1,False,United Kingdom,False,"British TV Shows, Reality TV",British TV Shows,2,TV-14,Teens,2021,2020s,2021-09-24,2021,3,9,September,Friday,0,False,<NA>,NaN,9
6,s10,Movie,The Starling,Theodore Melfi,1,False,United States,False,"Comedies, Dramas",Comedies,2,PG-13,Teens,2021,2020s,2021-09-24,2021,3,9,September,Friday,0,False,104,Long (90-120 min),<NA>
7,s939,Movie,Motu Patlu in the Game of Zones,Suhas Kadav,1,False,India,False,"Children & Family Movies, Comedies, Music & Mu...",Children & Family Movies,3,TV-Y7,Older Kids,2019,2010s,2021-05-01,2021,2,5,May,Saturday,2,False,87,Medium (60-90 min),<NA>
8,s13,Movie,Je Suis Karl,Christian Schwochow,1,False,Germany,False,"Dramas, International Movies",Dramas,2,TV-MA,Adults,2021,2020s,2021-09-23,2021,3,9,September,Thursday,0,False,127,Very long (>120 min),<NA>
9,s940,Movie,Motu Patlu in Wonderland,Suhas Kadav,1,False,India,False,"Children & Family Movies, Music & Musicals",Children & Family Movies,2,TV-Y7,Older Kids,2013,2010s,2021-05-01,2021,2,5,May,Saturday,8,False,76,Medium (60-90 min),<NA>


In [28]:
# Quick sanity analysis to prove the data is analysis-ready
print(df["type"].value_counts(), "\n")
print(df.groupby("year_added").size().tail(8), "\n")
print(df["country"].value_counts().head(5), "\n")
print(df["primary_genre"].value_counts().head(5))

type
Movie      6123
TV Show    2663
Name: count, dtype: int64 

year_added
2014      24
2015      82
2016     426
2017    1185
2018    1647
2019    2013
2020    1879
2021    1498
dtype: int64 

country
United States     3240
India             1056
United Kingdom     638
Pakistan           420
Unknown            287
Name: count, dtype: int64 

primary_genre
Dramas                    1597
Comedies                  1209
Action & Adventure         859
Documentaries              829
International TV Shows     772
Name: count, dtype: int64


In [29]:
# Save the cleaned dataset
OUTPUT_PATH = "netflix_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} columns -> {OUTPUT_PATH}")

Saved 8,786 rows x 26 columns -> netflix_cleaned.csv


## Thank you